<h1 style="text-align: center;">Main Pipeline</h1>

<p style="text-align: center;">
  1. Load preprocessed movie dataset.<br>
  2. Apply text stemming.<br>
  3. Convert text into numerical vectors using <code>CountVectorizer</code>.<br>
  4. Calculate cosine similarity between vectors.<br>
  5. Pass a movie into a recommendation function to return the top 5 similar movies.
</p>

<p style="text-align: center;">
  <b>Recommendation Logic : </b> Selected movie is mapped as a vector. Movies with the 5 highest cosine similarity scores are recommended.<br>
  <i>Note: Uses vector similarity instead of a traditional trained ML model.</i>
</p>

In [1]:
import pandas as pd
import numpy as np

In [2]:
new_df = pd.read_csv("Dataset/new_df.csv")

In [3]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [4]:
new_df['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [5]:
new_df['tags'][1]

"captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems. adventure fantasy action ocean drugabuse exoticisland eastindiatradingcompany loveofone'slife traitor shipwreck strongwoman ship alliance calypso afterlife fighter pirate swashbuckler aftercreditsstinger johnnydepp orlandobloom keiraknightley goreverbinski"

- ### Apply `Stemming` to  the `tags` column.   So, ['accepted', 'accepts', 'accepts'] -> ['accept','accept', 'accept']

In [6]:
import nltk

In [7]:
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()

In [8]:
stemmer.stem('accepted')

'accept'

In [9]:
def stem(text):
    y = []   # Define a empty list

    for i in text.split():     # `text.spilit()` String converted to a list
        y.append(stemmer.stem(i))

    return " ".join(y)      # Again convert List to String

In [10]:
stem('in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron')

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [11]:
new_df['tags'].apply(stem)

0       in the 22nd century, a parapleg marin is dispa...
1       captain barbossa, long believ to be dead, ha c...
2       a cryptic messag from bond’ past send him on a...
3       follow the death of district attorney harvey d...
4       john carter is a war-weary, former militari ca...
                              ...                        
4801    el mariachi just want to play hi guitar and ca...
4802    a newlyw couple' honeymoon is upend by the arr...
4803    "signed, sealed, delivered" introduc a dedic q...
4804    when ambiti new york attorney sam is sent to s...
4805    ever sinc the second grade when he first saw h...
Name: tags, Length: 4806, dtype: object

In [12]:
new_df['tags'] = new_df['tags'].apply(stem)

In [13]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


---------

- ## Vectorization

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000, stop_words='english')   # `max_features` define how many words I keepp.

In [15]:
vectors = vectorizer.fit_transform(new_df['tags']).toarray()

In [16]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [17]:
vectors.shape

(4806, 5000)

In [18]:
words = vectorizer.get_feature_names_out()
print(words[:100])       # See first 100 words  

['000' '007' '10' '100' '11' '12' '13' '14' '15' '16' '17' '17th' '18'
 '18th' '18thcenturi' '19' '1910' '1920' '1930' '1940' '1944' '1950'
 '1950s' '1960' '1960s' '1970' '1970s' '1971' '1974' '1976' '1980' '1985'
 '1990' '1999' '19th' '19thcenturi' '20' '200' '2003' '2009' '20th' '21st'
 '23' '24' '25' '30' '300' '3d' '40' '50' '500' '60' '70' '80' 'aaron'
 'aaroneckhart' 'abandon' 'abduct' 'abigailbreslin' 'abil' 'abl' 'aboard'
 'abov' 'abus' 'academ' 'academi' 'accept' 'access' 'accid' 'accident'
 'acclaim' 'accompani' 'accomplish' 'account' 'accus' 'ace' 'achiev'
 'acquaint' 'act' 'action' 'actionhero' 'activ' 'activist' 'activities'
 'actor' 'actress' 'actual' 'ad' 'adam' 'adamsandl' 'adamshankman' 'adapt'
 'add' 'addict' 'adjust' 'admir' 'admit' 'adolesc' 'adopt' 'ador']


----------

- ## Co-sine Distance

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
similarity = cosine_similarity(vectors)

In [21]:
similarity

array([[1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
        0.        ],
       [0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
        0.02615329],
       [0.0860309 , 0.06063391, 1.        , ..., 0.02451452, 0.        ,
        0.        ],
       ...,
       [0.04499213, 0.02378257, 0.02451452, ..., 1.        , 0.03962144,
        0.04229549],
       [0.        , 0.        , 0.        , ..., 0.03962144, 1.        ,
        0.08714204],
       [0.        , 0.02615329, 0.        , ..., 0.04229549, 0.08714204,
        1.        ]])

In [22]:
similarity.shape

(4806, 4806)

In [23]:
similarity[0]

array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
       0.        ])

---------

- ### Now create a function ,in this function pass one movie then this function return 5 similar movies name.

In [24]:
def recommend(movie):
    # Find the index of the movie whose title matches 'movie'
    movie_index = new_df[new_df['title'] == movie].index[0]        

    # Get the similarity scores of the selected movie with all movies.
    distances = similarity[movie_index]    
    # Sort movies by similarity score (highest to lowest) and select the top 5 similar movies
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:6] 

    for i in movies_list:
        # Get the title of the recommended movie and print it.
        print(new_df.iloc[i[0]].title)

In [25]:
recommend('Batman Begins')

The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


-----------

In [26]:
import pickle

In [27]:
pickle.dump(new_df,open('Dataset/movies.pkl','wb'))

In [28]:
pickle.dump(similarity,open('Dataset/similarity.pkl','wb'))